In [1]:
!git clone --single-branch --branch copilot/make-code-async https://github.com/timofeykhodykin/ai360-financial-qa.git

Cloning into 'ai360-financial-qa'...
remote: Enumerating objects: 767, done.
remote: Counting objects: 100% (216/216), done.
remote: Compressing objects: 100% (172/172), done.
remote: Total 767 (delta 75), reused 150 (delta 42), pack-reused 551 (from 1)
Receiving objects: 100% (767/767), 192.34 MiB | 41.82 MiB/s, done.
Resolving deltas: 100% (287/287), done.
Updating files: 100% (221/221), done.


In [2]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [3]:
import sys

In [4]:
sys.path.append('/kaggle/working/ai360-financial-qa/')

In [5]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers import SlidingWindowChunker
from financial_qa.embedders import HFEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop
from financial_qa.evaluation import evaluate_async, load_jsonl

In [6]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
OPENROUTER_API_KEY = user_secrets.get_secret("OPENROUTER_API_KEY")

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [ ]:
dotenv.load_dotenv('.env')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY is required')

DATASET_FILE = '/kaggle/working/ai360-financial-qa/dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = None

RAG_DB = 'gigar_embed'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
TOP_K = 5
EMBED_MODEL = 'ai-sage/Giga-Retrieval-instruct'

GEN_MODEL = 'google/gemini-2.0-flash-lite-001'
QUERY_CONCURRENCY = 4

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
JUDGE_PROCESSES = None  # None => one process per question

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download

repo_path = snapshot_download(
    "ai-sage/Giga-Retrieval-instruct",
    ignore_patterns=["*.safetensors", "*.bin", "*.pt"],
)

patches = {
    "modeling_gigarembed.py": [
        (
            "from transformers.models.llama.modeling_llama import LLAMA_INPUTS_DOCSTRING\n",
            "",
        ),
        (
            "@add_start_docstrings_to_model_forward(LLAMA_INPUTS_DOCSTRING)",
            '@add_start_docstrings_to_model_forward("")',
        ),
        (
            '        self._attn_implementation = "eager"\n\n    def forward(self, hiddens',
            '        self._attn_implementation = "eager"\n        self.post_init()\n\n    def forward(self, hiddens',
        ),
        (
            "            self.add_pad_token()\n\n    def add_pad_token",
            "            self.add_pad_token()\n        self.post_init()\n\n    def add_pad_token",
        ),
    ],
    "configuration_gigarembed.py": [
        (
            "        self.cross_dim_head = cross_dim_head\n        self._attn_implementation",
            "        self.cross_dim_head = cross_dim_head\n        super().__init__(**kwargs)\n        self._attn_implementation",
        ),
    ],
}

for filename, replacements in patches.items():
    path = os.path.join(repo_path, filename)
    code = open(path).read()
    for old, new in replacements:
        code = code.replace(old, new)
    open(path, "w").write(code)

# modules_dir = os.path.expanduser("~/.cache/huggingface/modules/transformers_modules")
# for entry in os.listdir(modules_dir):
#     if "sage" in entry.lower() or "giga" in entry.lower():
#         shutil.rmtree(os.path.join(modules_dir, entry))

print("Patched.")

In [ ]:
hf_embedder_code = """
import asyncio
import torch
from financial_qa.base import BaseEmbedder
from sentence_transformers import SentenceTransformer
from transformers import AutoModel, BitsAndBytesConfig
from typing import List


class HFEmbedder(BaseEmbedder):
    def __init__(self, model_name: str = "deepvk/USER2-base"):
        self.model_name = model_name
        native = AutoModel.from_pretrained(
            model_name,
            trust_remote_code=True,
            quantization_config=BitsAndBytesConfig(
                load_in_8bit=True,
                bnb_8bit_compute_dtype=torch.float16,
            ),
            device_map="auto",
        )
        if hasattr(native, "encode") and callable(native.encode):
            self._model = native
            self._is_native = True
        else:
            del native
            self._model = SentenceTransformer(model_name, trust_remote_code=True)
            self._is_native = False

    def embed(self, text: str) -> List[float]:
        if self._is_native:
            return self._model.encode([text])[0].cpu().tolist()
        return self._model.encode(text).tolist()

    def embed_passages(self, texts: List[str], batch_size: int = 4) -> List[List[float]]:
        if self._is_native:
            results = []
            for i in range(0, len(texts), batch_size):
                batch = self._model.encode(texts[i : i + batch_size])
                results.extend(batch.cpu().tolist())
            return results
        return self._model.encode(texts, batch_size=batch_size).tolist()

    async def aembed(self, text: str) -> List[float]:
        return await asyncio.to_thread(self.embed, text)

    async def aembed_passages(self, texts: List[str]) -> List[List[float]]:
        return await asyncio.gather(*[asyncio.to_thread(self.embed, t) for t in texts])

    def update(self, model_name: str) -> None:
        if model_name != self.model_name:
            self.__init__(model_name)
"""

with open("ai360-financial-qa/embedders/hf_embedder.py", "w") as f:
    f.write(hf_embedder_code)

In [8]:
all_records = load_jsonl(DATASET_FILE)

In [9]:
len(all_records)

449

In [10]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records 
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
    else:
        print('huy')
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))


449
Loaded 449 records (split=None)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


In [11]:
chunker = SlidingWindowChunker(chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
embedder = HFEmbedder(model_name=EMBED_MODEL)
rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/parsed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
)

store_dir = Path('/kaggle/working/ai360-financial-qa/indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc()
else:
    print(f'Using existing index at {store_dir}')


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Using existing index at /kaggle/working/ai360-financial-qa/indexes/default


In [12]:
loop = OpenRouterAgentLoop(rag=rag, model=GEN_MODEL)
print(f'Agent log file: {loop.log_path}')

Agent log file: logs/agent/agent_actions_20260519_232849_57_2bc5bb90.jsonl


In [13]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')


Querying agent: 100%|██████████| 449/449 [03:21<00:00,  2.22question/s, avg_s=1.79, errors=0, last_conf=0.889]

Done: 449 answers, 0 errors


In [14]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    api_key=OPENROUTER_API_KEY,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 449/449 [00:03<00:00, 121.04question/s, accuracy=38.75%, correct=174, errors=0]


In [15]:
result['correct'] / result['total']

0.38752783964365256

In [20]:
for res in result['results'][0:3]:
    print(res)
    print('-' * 75)

{'question_id': 'q_009c97884b8dc010', 'question': 'Каковы чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года?', 'gold_answer': '5 592 млн руб.', 'predicted_answer': 'Чистые комиссионные доходы Банка ДомРФ за шесть месяцев, закончившихся 30 июня 2025 года, составили 5 592 млн. рублей.\n', 'judge_score': 1, 'judge_reasoning': 'Оба ответа содержат одинаковую сумму чистых комиссионных доходов.'}
---------------------------------------------------------------------------
{'question_id': 'q_00d660efcf3e4607', 'question': 'Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?', 'gold_answer': '1,151 тыс. белорусских рублей', 'predicted_answer': 'Я не могу ответить на этот вопрос, так как в предоставленных фрагментах нет информации об общей сумме финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые став